In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
df_silver = spark.table("workspace.default.capstone_silver_sales")

In [0]:
df_gold_metrics = df_silver.groupBy("year", "month", "month_name", "state", "category").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("quantity").alias("units_sold"),
    F.sum("gross_amount").cast(T.DecimalType(18, 2)).alias("gross_sales"),
    F.sum("discount_amount").cast(T.DecimalType(18, 2)).alias("discount_amount"),
    F.sum("net_amount").cast(T.DecimalType(18, 2)).alias("net_sales")
).withColumn(
    # Compute AOV by dividing net_sales by total_orders
    "avg_order_value", 
    (F.col("net_sales") / F.col("total_orders")).cast(T.DecimalType(18, 2))
)
display(df_gold_metrics)

df_gold_metrics.write.mode("overwrite").saveAsTable("workspace.default.capstone_gold_sales_summary")